# Polyak 和 Nesterov 的 long-time loss curve

这里不固定 data budget $D=BK$，而是直接延长训练时间，观察 loss 从 signal-dominated decay 进入 noise-dominated plateau：

- **实线**：数值模拟；
- **虚线**：scaling-law prediction。

Polyak 和 Nesterov 放在两个独立运行单元格中；完成公共初始化后，可以只运行其中一个。两者的结果在后续单元格中叠加到同一张图上：真实曲线使用浅色半透明线，理论预测使用不透明实线。Notebook 完全独立，只依赖 NumPy、PyTorch、Matplotlib 和 tqdm。

In [ ]:
from dataclasses import dataclass
from itertools import product
import json
import math
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

# ===== Long-time experiment =====
SEED = 0
DEVICE = "auto"       # "auto", "cpu", "cuda", "cuda:0", or "mps"

BETA = 4.0
S = 0.3
SIGMA = 0.5
DIM = 4096
RUNS = 100
TRAIN_STEPS = 500_000
SHOW_PROGRESS = True
SAVE_EVERY_STEP = True  # True: 保存每一个 optimization step 的 loss
OUTPUT_ROOT = Path("scaling_law_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# 这组参数让 crossover 落在可观察的训练区间内：
# Polyak: transient -> scaling -> noise
# Nesterov: scaling -> noise；其 transient 被理论 prefactor 强烈压低，不会主导。
SETTINGS = {
    "polyak":   {"B": 64, "eta": 0.1, "rho": 0.995},
    "nesterov": {"B": 64,   "eta": 0.1, "rho": 0.995},
}


## 模型和 simulation

使用原 setting 的 power-law spectrum、target smoothness、label noise 和 curvature-aligned Gaussian gradient noise：

$$
\lambda_j=j^{-\beta},\qquad
\lambda_j|\theta_j^*|^2=j^{-(1+s\beta)}.
$$

每个算法运行固定的 `TRAIN_STEPS`，所以总数据量不再被预先固定。

In [ ]:
def get_device(name):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass
class Problem:
    lam: torch.Tensor
    theta_star: torch.Tensor
    sigma: float


def build_problem(dim, beta, s, sigma, device, dtype):
    j = torch.arange(1, dim + 1, device=device, dtype=dtype)
    lam = j.pow(-beta)
    theta_star = j.pow(-0.5 * (1.0 + (s - 1.0) * beta))
    return Problem(lam, theta_star, sigma)


def loss(delta, lam):
    return 0.5 * torch.sum(lam * delta.square(), dim=-1)


@torch.no_grad()
def simulate(
    problem, algorithm, B, eta, rho, steps, runs, seed,
    replica_loss_path=None, save_every_step=None,
):
    if save_every_step is None:
        save_every_step = SAVE_EVERY_STEP

    generator = torch.Generator(device=problem.lam.device)
    generator.manual_seed(seed)

    delta = -problem.theta_star.expand(runs, -1).clone()
    previous = delta.clone()

    if save_every_step:
        checkpoints = None
    else:
        checkpoints = set(range(min(steps, 100) + 1))
        checkpoints.update(np.geomspace(1, steps, 500).round().astype(int).tolist())
        checkpoints.add(steps)

    initial_replica_losses = loss(delta, problem.lam)
    saved_steps = [0]
    saved_losses = [initial_replica_losses.mean().item()]

    # 完整 replica-level loss 逐步写入 .npy，避免把全部数据留在内存中。
    replica_losses = None
    if save_every_step and replica_loss_path is not None:
        replica_loss_path = Path(replica_loss_path)
        replica_loss_path.parent.mkdir(parents=True, exist_ok=True)
        replica_losses = np.lib.format.open_memmap(
            replica_loss_path,
            mode="w+",
            dtype=np.float64,
            shape=(steps + 1, runs),
        )
        replica_losses[0] = initial_replica_losses.detach().cpu().numpy()

    iterator = tqdm(
        range(1, steps + 1),
        total=steps,
        desc=f"{algorithm.capitalize()} training",
        unit="step",
        mininterval=0.5,
        disable=not SHOW_PROGRESS,
    )

    for k in iterator:
        momentum = delta - previous
        evaluation = delta + rho * momentum if algorithm == "nesterov" else delta

        evaluation_loss = loss(evaluation, problem.lam)
        scale = ((evaluation_loss + problem.sigma**2) / B).sqrt().unsqueeze(-1)
        noise = torch.randn(
            evaluation.shape, device=evaluation.device,
            dtype=evaluation.dtype, generator=generator,
        )
        gradient = evaluation * problem.lam + noise * problem.lam.sqrt() * scale

        new_delta = delta + rho * momentum - eta * gradient
        previous, delta = delta, new_delta

        if save_every_step or k in checkpoints:
            current_replica_losses = loss(delta, problem.lam)
            saved_steps.append(k)
            saved_losses.append(current_replica_losses.mean().item())
            if replica_losses is not None:
                replica_losses[k] = current_replica_losses.detach().cpu().numpy()

    if replica_losses is not None:
        replica_losses.flush()
        del replica_losses

    return np.asarray(saved_steps), np.asarray(saved_losses)


## Scaling-law prediction

定义 $\eta_\rho=\eta/(1-\rho)$。理论只给出 $\asymp$：

$$
L_{\rm Polyak}(k)
\asymp
\rho^k+(\eta_\rho k)^{-s}+\frac{\sigma^2}{B}\eta_\rho,
$$

$$
L_{\rm Nesterov}(k)
\asymp
\left(\frac{(1-\rho)^2}{\eta}\right)^{s+1/\beta}
\left(\frac{4\rho^2}{(1+\rho)^2}\right)^k
+(\eta_\rho k)^{-s}
+\frac{\sigma^2}{B}\min\{\eta_\rho,\eta_\rho^{1/\beta}\}.
$$

因此数值预测写成

$$
L_{\rm pred}(k)=C_T T(k)+C_S S(k)+C_N N(k),
\qquad C_T,C_S,C_N\ge0.
$$

三个 $O(1)$ 系数用 simulation curve 做非负 relative least-squares 拟合。它们只用于恢复 $\asymp$ 中没有给出的常数；最终图仍然只画一条总 prediction 虚线，不展示 component curves。

有效的 scaling-to-noise balance point 由拟合后的两项相等定义：

$$
C_S(\eta_\rho k_*)^{-s}=C_NN.
$$

这比假设 $C_S=C_N=1$ 得到的 unit-coefficient scale 更可靠。由于两项之和是光滑的，$k_*$ 是 balance scale，不是尖锐折点。


In [ ]:
def theory_matrix(steps, algorithm, B, eta, rho, *, beta=None, s=None, sigma=None):
    beta = BETA if beta is None else beta
    s = S if s is None else s
    sigma = SIGMA if sigma is None else sigma
    k = np.maximum(np.asarray(steps, dtype=float), 1.0)
    eta_rho = eta / (1.0 - rho)
    signal = (eta_rho * k) ** (-s)

    if algorithm == "polyak":
        transient = rho**k
        noise = np.full_like(k, sigma**2 * eta_rho / B)
    else:
        r_nes = 4.0 * rho**2 / (1.0 + rho)**2
        transient = ((1.0 - rho)**2 / eta) ** (s + 1.0 / beta) * r_nes**k
        floor = sigma**2 / B * min(eta_rho, eta_rho ** (1.0 / beta))
        noise = np.full_like(k, floor)

    return np.column_stack([transient, signal, noise])


def nonnegative_lstsq(X, y):
    # Only three variables: enumerate all nonempty active sets, avoiding SciPy.
    best_coef = np.zeros(X.shape[1])
    best_error = np.inf
    for mask in range(1, 1 << X.shape[1]):
        active = [j for j in range(X.shape[1]) if mask & (1 << j)]
        coef_active, *_ = np.linalg.lstsq(X[:, active], y, rcond=None)
        if np.any(coef_active < 0):
            continue
        coef = np.zeros(X.shape[1])
        coef[active] = coef_active
        error = np.mean((X @ coef - y) ** 2)
        if error < best_error:
            best_coef, best_error = coef, error
    return best_coef


def fit_prediction(
    steps, empirical, algorithm, B, eta, rho, *, beta=None, s=None, sigma=None
):
    X = theory_matrix(
        steps, algorithm, B, eta, rho, beta=beta, s=s, sigma=sigma
    )
    eta_rho = eta / (1.0 - rho)
    valid = (steps > 0) & (eta_rho * steps >= 1.0)

    # Relative least squares: each log-time region has comparable influence.
    scale = np.maximum(empirical[valid], np.finfo(float).tiny)
    coefficients = nonnegative_lstsq(X[valid] / scale[:, None], np.ones(valid.sum()))
    prediction = X @ coefficients
    return prediction, valid, coefficients


def fitted_transition_predictions(
    algorithm, B, eta, rho, coefficients, *, beta=None, s=None, sigma=None,
    train_steps=None,
):
    # 保留该诊断函数供需要时使用；绘图中不展示转折点。
    beta = BETA if beta is None else beta
    s = S if s is None else s
    sigma = SIGMA if sigma is None else sigma
    train_steps = TRAIN_STEPS if train_steps is None else train_steps

    c_transient, c_signal, c_noise = coefficients
    eta_rho = eta / (1.0 - rho)
    X_one = theory_matrix(
        np.array([1.0]), algorithm, B, eta, rho,
        beta=beta, s=s, sigma=sigma,
    )[0]
    noise_shape = X_one[2]

    if c_signal > 0 and c_noise > 0 and noise_shape > 0:
        scaling_to_noise = (c_signal / (c_noise * noise_shape)) ** (1.0 / s) / eta_rho
    else:
        scaling_to_noise = None

    if c_transient <= 0 or c_signal <= 0:
        transient_to_scaling = None
        max_ratio = 0.0
    else:
        if algorithm == "polyak":
            decay_base = rho
            prefactor = 1.0
        else:
            decay_base = 4.0 * rho**2 / (1.0 + rho)**2
            prefactor = ((1.0 - rho)**2 / eta) ** (s + 1.0 / beta)

        decay_rate = -np.log(decay_base)
        k_at_max = max(1.0, s / decay_rate)
        max_ratio = (
            c_transient * prefactor * decay_base**k_at_max
            / (c_signal * (eta_rho * k_at_max) ** (-s))
        )

        grid = np.geomspace(1.0, max(float(train_steps), 10.0), 20_000)
        log_ratio = (
            np.log(c_transient * prefactor / c_signal)
            + grid * np.log(decay_base)
            + s * np.log(eta_rho * grid)
        )
        crossings = np.flatnonzero(log_ratio[:-1] * log_ratio[1:] < 0)
        if crossings.size:
            i = crossings[-1]
            left, right = grid[i], grid[i + 1]
            for _ in range(80):
                middle = 0.5 * (left + right)
                value = (
                    np.log(c_transient * prefactor / c_signal)
                    + middle * np.log(decay_base)
                    + s * np.log(eta_rho * middle)
                )
                left_value = (
                    np.log(c_transient * prefactor / c_signal)
                    + left * np.log(decay_base)
                    + s * np.log(eta_rho * left)
                )
                if left_value * value <= 0:
                    right = middle
                else:
                    left = middle
            transient_to_scaling = 0.5 * (left + right)
        else:
            transient_to_scaling = None

    return {
        "transient_to_scaling": transient_to_scaling,
        "scaling_to_noise": scaling_to_noise,
        "max_transient_signal_ratio": max_ratio,
    }


## 公共初始化、参数目录和单算法运行函数

先运行这个单元格。它只初始化问题和保存函数，不会启动训练。

本次实验的所有 long-time 与 blow-up 数据统一放在一个参数标记目录中：

```text
scaling_law_outputs/beta-..._s-..._sigma-..._dim-..._runs-..._steps-.../
```

每个算法的文件名继续标记 `B`、`eta` 和 `rho`。因此修改参数重跑时不会与其他参数配置混淆。单算法运行函数只负责训练和保存数据；合并图在后面的独立单元格生成。


In [ ]:
def parameter_tag(value):
    """生成适合目录/文件名的稳定参数字符串。"""
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    return f"{float(value):.12g}".replace("-", "m").replace(".", "p").replace("+", "")


def algorithm_tag(algorithm):
    q = SETTINGS[algorithm]
    return (
        f"{algorithm}"
        f"_B-{parameter_tag(q['B'])}"
        f"_eta-{parameter_tag(q['eta'])}"
        f"_rho-{parameter_tag(q['rho'])}"
    )


EXPERIMENT_TAG = (
    f"beta-{parameter_tag(BETA)}"
    f"_s-{parameter_tag(S)}"
    f"_sigma-{parameter_tag(SIGMA)}"
    f"_dim-{parameter_tag(DIM)}"
    f"_runs-{parameter_tag(RUNS)}"
    f"_steps-{parameter_tag(TRAIN_STEPS)}"
)
EXPERIMENT_DIR = OUTPUT_ROOT / EXPERIMENT_TAG
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

seed_everything(SEED)
device = get_device(DEVICE)
dtype = torch.float32 if device.type == "mps" else torch.float64
problem = build_problem(DIM, BETA, S, SIGMA, device, dtype)

experiment_config = {
    "experiment_tag": EXPERIMENT_TAG,
    "output_directory": str(EXPERIMENT_DIR),
    "seed": SEED,
    "device_requested": DEVICE,
    "device_used": str(device),
    "dtype": str(dtype),
    "beta": BETA,
    "s": S,
    "sigma": SIGMA,
    "dim": DIM,
    "runs": RUNS,
    "train_steps": TRAIN_STEPS,
    "save_every_step": SAVE_EVERY_STEP,
    "settings": SETTINGS,
}
with (EXPERIMENT_DIR / "experiment_config.json").open("w", encoding="utf-8") as f:
    json.dump(experiment_config, f, indent=2, ensure_ascii=False)

print(f"device: {device}; dtype: {dtype}")
print("all experiment data will be saved in:")
print(EXPERIMENT_DIR.resolve())


def run_long_time_experiment(algorithm, seed):
    """独立运行并保存一个算法的 long-time experiment。"""
    q = SETTINGS[algorithm]
    tag = algorithm_tag(algorithm)
    t0 = time.time()

    replica_loss_path = EXPERIMENT_DIR / f"{tag}_replica_losses.npy"
    steps, empirical = simulate(
        problem, algorithm, q["B"], q["eta"], q["rho"],
        TRAIN_STEPS, RUNS, seed,
        replica_loss_path=replica_loss_path,
    )
    prediction, valid, coefficients = fit_prediction(
        steps, empirical, algorithm, q["B"], q["eta"], q["rho"]
    )
    theory_terms = theory_matrix(
        steps, algorithm, q["B"], q["eta"], q["rho"]
    )

    result = {
        "steps": steps,
        "empirical": empirical,
        "prediction": prediction,
        "valid": valid,
        "theory_terms": theory_terms,
        "coefficients": coefficients,
        "replica_loss_path": replica_loss_path,
    }

    output_path = EXPERIMENT_DIR / f"{tag}_results.npz"
    np.savez_compressed(
        output_path,
        steps=steps,
        empirical_loss=empirical,
        theory_prediction=prediction,
        theory_transient=theory_terms[:, 0],
        theory_scaling=theory_terms[:, 1],
        theory_noise=theory_terms[:, 2],
        fit_valid_mask=valid,
        fitted_coefficients=coefficients,
        beta=np.asarray(BETA),
        s=np.asarray(S),
        sigma=np.asarray(SIGMA),
        dim=np.asarray(DIM),
        runs=np.asarray(RUNS),
        train_steps=np.asarray(TRAIN_STEPS),
        B=np.asarray(q["B"]),
        eta=np.asarray(q["eta"]),
        rho=np.asarray(q["rho"]),
        seed=np.asarray(seed),
    )

    algorithm_config = {
        **experiment_config,
        "algorithm": algorithm,
        "algorithm_tag": tag,
        "algorithm_seed": seed,
        "B": q["B"],
        "eta": q["eta"],
        "rho": q["rho"],
        "results_file": output_path.name,
        "replica_losses_file": replica_loss_path.name,
    }
    config_path = EXPERIMENT_DIR / f"{tag}_config.json"
    with config_path.open("w", encoding="utf-8") as f:
        json.dump(algorithm_config, f, indent=2, ensure_ascii=False)

    print(f"\n{algorithm.upper()}")
    print(f"B={q['B']}, eta={q['eta']}, rho={q['rho']}, steps={TRAIN_STEPS:,}")
    print(
        "fitted constants [transient, scaling, noise] = "
        + np.array2string(coefficients, precision=4)
    )
    print(f"saved mean curve and theory: {output_path}")
    print(f"saved all replica losses:    {replica_loss_path}")
    print(f"saved parameter config:      {config_path}")
    print(f"simulation time: {time.time()-t0:.1f}s")
    return result


## 单独运行 Polyak

只想运行 Polyak 时，在完成上面的公共初始化后，仅运行这个单元格。该单元只训练并保存 Polyak 数据，不单独绘图。


In [ ]:
polyak_result = run_long_time_experiment(
    "polyak",
    seed=SEED,
)


## 单独运行 Nesterov

只想运行 Nesterov 时，在完成上面的公共初始化后，仅运行这个单元格。该单元只训练并保存 Nesterov 数据，不单独绘图。


In [ ]:
nesterov_result = run_long_time_experiment(
    "nesterov",
    seed=SEED + 10_000,
)


## 将 Polyak 和 Nesterov 画在同一张图上

运行两个算法后执行这个单元格。若当前 kernel 中缺少某个结果变量，会尝试从当前参数目录读取已经保存的结果。

- **浅色半透明线**：simulation；
- **深色实线**：theory prediction。


In [ ]:
def load_saved_long_time_result(algorithm):
    """读取当前参数目录中某个算法已经保存的汇总结果。"""
    path = EXPERIMENT_DIR / f"{algorithm_tag(algorithm)}_results.npz"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run the {algorithm.capitalize()} cell first."
        )
    with np.load(path) as data:
        return {
            "steps": data["steps"].copy(),
            "empirical": data["empirical_loss"].copy(),
            "prediction": data["theory_prediction"].copy(),
            "valid": data["fit_valid_mask"].astype(bool),
            "theory_terms": np.column_stack([
                data["theory_transient"],
                data["theory_scaling"],
                data["theory_noise"],
            ]),
            "coefficients": data["fitted_coefficients"].copy(),
        }


plot_results = {
    "polyak": globals().get("polyak_result"),
    "nesterov": globals().get("nesterov_result"),
}
for algorithm in plot_results:
    if plot_results[algorithm] is None:
        plot_results[algorithm] = load_saved_long_time_result(algorithm)

colors = {
    "polyak": {"empirical": "#f6bd83", "prediction": "#d55e00"},
    "nesterov": {"empirical": "#9ecae1", "prediction": "#0072b2"},
}

fig, ax = plt.subplots(figsize=(3.25, 2.45))
for algorithm in ("polyak", "nesterov"):
    result = plot_results[algorithm]
    q = SETTINGS[algorithm]
    steps = result["steps"]
    positive = steps > 0
    valid = result["valid"]
    name = algorithm.capitalize()
    parameter_label = f"B={q['B']}, eta={q['eta']}, rho={q['rho']}"

    ax.plot(
        steps[positive], result["empirical"][positive],
        color=colors[algorithm]["empirical"],
        alpha=0.55,
        lw=1.4,
        label="_nolegend_",
        zorder=1,
    )
    ax.plot(
        steps[valid], result["prediction"][valid],
        color=colors[algorithm]["prediction"],
        ls="-",
        lw=1.8,
        label=name,
        zorder=3,
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Step", fontsize=9)
ax.set_ylabel("Loss", fontsize=9)
ax.tick_params(axis="both", which="major", labelsize=8)
ax.tick_params(axis="both", which="minor", labelsize=7)
ax.legend(fontsize=8, frameon=False, handlelength=1.8, borderaxespad=0.3)
fig.tight_layout()

combined_figure_path = EXPERIMENT_DIR / (
    f"combined_{algorithm_tag('polyak')}__{algorithm_tag('nesterov')}.pdf"
)
fig.savefig(combined_figure_path, dpi=200, bbox_inches="tight")
print(f"saved combined figure to {combined_figure_path}")
plt.show()


# 100 组 $(\beta,s,\sigma)$ 参数网格实验

网格为：

- $\beta\in\{2,3,4,6,8\}$；
- $s\in\{0.3,0.5,0.7,1,2\}$；
- $\sigma\in\{0.1,0.3,0.5,1\}$。

共 $5\times5\times4=100$ 组。每组沿用上方 `SETTINGS` 中 Polyak 和 Nesterov 各自的 $B,\eta,\rho$，运行两个算法并生成一张合并图。

默认保存每一步、每个 replica 的 loss。按当前 `TRAIN_STEPS=500000`、`RUNS=100` 和 `float64` 计算，仅 replica loss 数据约占 **80 GB**。批量运行支持断点续跑，已完整保存的算法会自动跳过。


In [ ]:
BETA_GRID = [2.0, 3.0, 4.0, 6.0, 8.0]
S_GRID = [0.3, 0.5, 0.7, 1.0, 2.0]
SIGMA_GRID = [0.1, 0.3, 0.5, 1.0]

GRID_OUTPUT_ROOT = OUTPUT_ROOT / "beta_s_sigma_grid_100"
GRID_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

GRID_SKIP_COMPLETED = True
GRID_SAVE_REPLICA_LOSSES = True
GRID_MAKE_COMBINED_PLOTS = True
GRID_SHOW_PLOTS = False  # 100 张图不在 notebook 中逐张展示，只保存 PDF

GRID_COMBINATIONS = list(product(BETA_GRID, S_GRID, SIGMA_GRID))
assert len(GRID_COMBINATIONS) == 100

# 两种算法使用不同但确定性的 seed；每个网格组合也有唯一 seed offset。
def grid_seed(combo_index, algorithm):
    algorithm_offset = 0 if algorithm == "polyak" else 10_000
    return SEED + combo_index * 100_000 + algorithm_offset


print(f"number of parameter combinations: {len(GRID_COMBINATIONS)}")
print(f"two algorithms per combination:   {2 * len(GRID_COMBINATIONS)} runs")
print(f"grid output root: {GRID_OUTPUT_ROOT.resolve()}")
if GRID_SAVE_REPLICA_LOSSES:
    estimated_bytes = (
        len(GRID_COMBINATIONS) * 2 * (TRAIN_STEPS + 1) * RUNS * 8
    )
    print(f"estimated replica-loss storage: {estimated_bytes / 1e9:.1f} GB")


## 运行或继续运行全部 100 组

执行下面的单元格即可开始批量实验。每个算法完成后立即落盘；如果中途停止，再次执行会从未完成的位置继续。


In [ ]:
def grid_experiment_tag(beta, s, sigma):
    return (
        f"beta-{parameter_tag(beta)}"
        f"_s-{parameter_tag(s)}"
        f"_sigma-{parameter_tag(sigma)}"
        f"_dim-{parameter_tag(DIM)}"
        f"_runs-{parameter_tag(RUNS)}"
        f"_steps-{parameter_tag(TRAIN_STEPS)}"
    )


def grid_algorithm_tag(algorithm):
    q = SETTINGS[algorithm]
    return (
        f"{algorithm}"
        f"_B-{parameter_tag(q['B'])}"
        f"_eta-{parameter_tag(q['eta'])}"
        f"_rho-{parameter_tag(q['rho'])}"
    )


def grid_paths(experiment_dir, algorithm):
    tag = grid_algorithm_tag(algorithm)
    return {
        "results": experiment_dir / f"{tag}_results.npz",
        "replicas": experiment_dir / f"{tag}_replica_losses.npy",
        "config": experiment_dir / f"{tag}_config.json",
    }


def grid_algorithm_complete(experiment_dir, algorithm):
    paths = grid_paths(experiment_dir, algorithm)
    required = [paths["results"], paths["config"]]
    if GRID_SAVE_REPLICA_LOSSES:
        required.append(paths["replicas"])
    if not all(path.exists() for path in required):
        return False

    try:
        with np.load(paths["results"]) as data:
            if data["steps"].shape != (TRAIN_STEPS + 1,):
                return False
        if GRID_SAVE_REPLICA_LOSSES:
            replicas = np.load(paths["replicas"], mmap_mode="r")
            if replicas.shape != (TRAIN_STEPS + 1, RUNS):
                return False
    except Exception:
        return False
    return True


def load_grid_result(experiment_dir, algorithm):
    path = grid_paths(experiment_dir, algorithm)["results"]
    with np.load(path) as data:
        return {
            "steps": data["steps"].copy(),
            "empirical": data["empirical_loss"].copy(),
            "prediction": data["theory_prediction"].copy(),
            "valid": data["fit_valid_mask"].astype(bool),
            "coefficients": data["fitted_coefficients"].copy(),
        }


def run_grid_algorithm(
    problem, algorithm, *, beta, s, sigma, combo_index, experiment_dir
):
    q = SETTINGS[algorithm]
    paths = grid_paths(experiment_dir, algorithm)
    seed = grid_seed(combo_index, algorithm)
    t0 = time.time()

    replica_path = paths["replicas"] if GRID_SAVE_REPLICA_LOSSES else None
    steps, empirical = simulate(
        problem,
        algorithm,
        q["B"],
        q["eta"],
        q["rho"],
        TRAIN_STEPS,
        RUNS,
        seed,
        replica_loss_path=replica_path,
        save_every_step=True,
    )
    prediction, valid, coefficients = fit_prediction(
        steps,
        empirical,
        algorithm,
        q["B"],
        q["eta"],
        q["rho"],
        beta=beta,
        s=s,
        sigma=sigma,
    )
    theory_terms = theory_matrix(
        steps,
        algorithm,
        q["B"],
        q["eta"],
        q["rho"],
        beta=beta,
        s=s,
        sigma=sigma,
    )

    # 临时文件写完后原子替换 results，避免中断时把半成品误判为完成。
    temporary_results = paths["results"].with_suffix(".tmp.npz")
    np.savez_compressed(
        temporary_results,
        steps=steps,
        empirical_loss=empirical,
        theory_prediction=prediction,
        theory_transient=theory_terms[:, 0],
        theory_scaling=theory_terms[:, 1],
        theory_noise=theory_terms[:, 2],
        fit_valid_mask=valid,
        fitted_coefficients=coefficients,
        beta=np.asarray(beta),
        s=np.asarray(s),
        sigma=np.asarray(sigma),
        dim=np.asarray(DIM),
        runs=np.asarray(RUNS),
        train_steps=np.asarray(TRAIN_STEPS),
        B=np.asarray(q["B"]),
        eta=np.asarray(q["eta"]),
        rho=np.asarray(q["rho"]),
        seed=np.asarray(seed),
    )
    temporary_results.replace(paths["results"])

    config = {
        "combo_index": combo_index,
        "algorithm": algorithm,
        "beta": beta,
        "s": s,
        "sigma": sigma,
        "dim": DIM,
        "runs": RUNS,
        "train_steps": TRAIN_STEPS,
        "B": q["B"],
        "eta": q["eta"],
        "rho": q["rho"],
        "seed": seed,
        "device": str(device),
        "dtype": str(dtype),
        "save_replica_losses": GRID_SAVE_REPLICA_LOSSES,
        "results_file": paths["results"].name,
        "replica_losses_file": paths["replicas"].name if replica_path else None,
        "runtime_seconds": time.time() - t0,
    }
    temporary_config = paths["config"].with_suffix(".tmp.json")
    with temporary_config.open("w", encoding="utf-8") as f:
        json.dump(config, f, indent=2, ensure_ascii=False)
    temporary_config.replace(paths["config"])

    return {
        "steps": steps,
        "empirical": empirical,
        "prediction": prediction,
        "valid": valid,
        "coefficients": coefficients,
    }


def save_grid_combined_plot(experiment_dir, beta, s, sigma, results):
    colors = {
        "polyak": {"empirical": "#f6bd83", "prediction": "#d55e00"},
        "nesterov": {"empirical": "#9ecae1", "prediction": "#0072b2"},
    }
    fig, ax = plt.subplots(figsize=(3.25, 2.45))
    for algorithm in ("polyak", "nesterov"):
        result = results[algorithm]
        q = SETTINGS[algorithm]
        steps = result["steps"]
        positive = steps > 0
        valid = result["valid"]
        name = algorithm.capitalize()
        ax.plot(
            steps[positive],
            result["empirical"][positive],
            color=colors[algorithm]["empirical"],
            alpha=0.55,
            lw=1.4,
            label="_nolegend_",
            zorder=1,
        )
        ax.plot(
            steps[valid],
            result["prediction"][valid],
            color=colors[algorithm]["prediction"],
            ls="-",
            lw=1.8,
            label=name,
            zorder=3,
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Step", fontsize=9)
    ax.set_ylabel("Loss", fontsize=9)
    ax.tick_params(axis="both", which="major", labelsize=8)
    ax.tick_params(axis="both", which="minor", labelsize=7)
    ax.legend(fontsize=8, frameon=False, handlelength=1.8, borderaxespad=0.3)
    fig.tight_layout()
    figure_path = experiment_dir / "combined_polyak_nesterov.pdf"
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    if GRID_SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)
    return figure_path


def write_grid_manifest(records):
    manifest = {
        "grid": {
            "beta": BETA_GRID,
            "s": S_GRID,
            "sigma": SIGMA_GRID,
            "number_of_combinations": len(GRID_COMBINATIONS),
        },
        "shared_parameters": {
            "dim": DIM,
            "runs": RUNS,
            "train_steps": TRAIN_STEPS,
            "settings": SETTINGS,
            "save_replica_losses": GRID_SAVE_REPLICA_LOSSES,
        },
        "experiments": records,
    }
    temporary = GRID_OUTPUT_ROOT / "grid_manifest.tmp.json"
    final = GRID_OUTPUT_ROOT / "grid_manifest.json"
    with temporary.open("w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    temporary.replace(final)


grid_records = []
grid_start = time.time()

for combo_index, (beta, s, sigma) in enumerate(GRID_COMBINATIONS):
    experiment_tag = grid_experiment_tag(beta, s, sigma)
    experiment_dir = GRID_OUTPUT_ROOT / experiment_tag
    experiment_dir.mkdir(parents=True, exist_ok=True)

    print(
        f"\n[{combo_index + 1:03d}/{len(GRID_COMBINATIONS)}] "
        f"beta={beta:g}, s={s:g}, sigma={sigma:g}"
    )

    current_problem = build_problem(DIM, beta, s, sigma, device, dtype)
    current_results = {}
    statuses = {}

    for algorithm in ("polyak", "nesterov"):
        complete = GRID_SKIP_COMPLETED and grid_algorithm_complete(
            experiment_dir, algorithm
        )
        if complete:
            print(f"  {algorithm}: complete, loading saved result")
            current_results[algorithm] = load_grid_result(experiment_dir, algorithm)
            statuses[algorithm] = "skipped_complete"
        else:
            print(f"  {algorithm}: running")
            current_results[algorithm] = run_grid_algorithm(
                current_problem,
                algorithm,
                beta=beta,
                s=s,
                sigma=sigma,
                combo_index=combo_index,
                experiment_dir=experiment_dir,
            )
            statuses[algorithm] = "completed"

    figure_path = None
    if GRID_MAKE_COMBINED_PLOTS:
        figure_path = save_grid_combined_plot(
            experiment_dir, beta, s, sigma, current_results
        )
        print(f"  combined plot: {figure_path.name}")

    record = {
        "combo_index": combo_index,
        "beta": beta,
        "s": s,
        "sigma": sigma,
        "directory": str(experiment_dir),
        "polyak_status": statuses["polyak"],
        "nesterov_status": statuses["nesterov"],
        "combined_plot": figure_path.name if figure_path else None,
    }
    grid_records.append(record)
    write_grid_manifest(grid_records)

    del current_problem, current_results
    if device.type == "cuda":
        torch.cuda.empty_cache()

print(f"\nfinished 100 combinations in {(time.time() - grid_start) / 3600:.2f} hours")
print(f"manifest: {GRID_OUTPUT_ROOT / 'grid_manifest.json'}")


## 汇总并下载全部实验图片

运行完网格实验后执行此单元格。它会把所有 `combined_polyak_nesterov.pdf` 集中复制到一个大文件夹，并打包成 ZIP。每张图片的文件名包含对应的 $\beta$、$s$、$\sigma$ 等参数，不会相互覆盖。


In [ ]:
import shutil

ALL_FIGURES_DIR = OUTPUT_ROOT / "all_grid_figures_pdf"
ALL_FIGURES_ZIP = OUTPUT_ROOT / "all_grid_figures_pdf.zip"

# 重新生成汇总目录，避免混入旧参数实验的图片。
if ALL_FIGURES_DIR.exists():
    shutil.rmtree(ALL_FIGURES_DIR)
ALL_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

figure_paths = sorted(
    GRID_OUTPUT_ROOT.glob("*/combined_polyak_nesterov.pdf")
)

for source_path in figure_paths:
    # 父目录名已经完整标记 beta、s、sigma、dim、runs 和 steps。
    destination_name = f"{source_path.parent.name}.pdf"
    shutil.copy2(source_path, ALL_FIGURES_DIR / destination_name)

if ALL_FIGURES_ZIP.exists():
    ALL_FIGURES_ZIP.unlink()

shutil.make_archive(
    str(ALL_FIGURES_ZIP.with_suffix("")),
    format="zip",
    root_dir=ALL_FIGURES_DIR.parent,
    base_dir=ALL_FIGURES_DIR.name,
)

print(f"collected figures: {len(figure_paths)} / {len(GRID_COMBINATIONS)}")
print(f"figure folder: {ALL_FIGURES_DIR.resolve()}")
print(f"download ZIP: {ALL_FIGURES_ZIP.resolve()}")
print(f"ZIP size: {ALL_FIGURES_ZIP.stat().st_size / 1024**2:.2f} MiB")

if len(figure_paths) != len(GRID_COMBINATIONS):
    print(
        "Warning: some experiments do not have a combined figure yet. "
        "Finish or resume the grid experiment, then rerun this cell."
    )
